# 02 — Do the candidate directions stay flat under parameter walks?

The Jacobian only gives a first-order test. This notebook reads the stored walk-test JSON and checks whether moving along candidate directions keeps the training loss nearly unchanged.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
print('repo root:', ROOT)


In [ ]:
def load(name):
    with open(RESULTS / name) as f:
        return json.load(f)

mlp = load('walk_test_mlp.json')
cnn = load('walk_test_cnn.json')
print('MLP anchors:', len(mlp))
print('CNN anchors:', len(cnn))


In [ ]:
def plot_walk(records, title):
    colors = {'gauge': '#5B8E7D', 'extra': '#2A6F97', 'curved': '#D97757', 'random': '#7B8085'}
    labels = {'gauge': 'known gauge', 'extra': 'extra subspace', 'curved': 'high-curvature', 'random': 'random'}
    eps = np.array(records[0]['walk_eps_grid'])
    fig, ax = plt.subplots(figsize=(7.2, 4))
    for key in ['gauge', 'extra', 'curved', 'random']:
        series = []
        for rec in records:
            item = rec['walk_results'].get(key, {'available': False})
            if not item.get('available'):
                continue
            losses = np.array(item['losses'])
            series.append(losses.mean(axis=1) - rec['L0'])
        if not series:
            continue
        arr = np.stack(series)
        ax.plot(eps, arr.mean(axis=0), '-o', color=colors[key], label=labels[key])
    ax.axhline(0, color='black', lw=0.6)
    ax.set_yscale('symlog', linthresh=1e-4)
    ax.set_xlabel('step size epsilon')
    ax.set_ylabel('change in training loss')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()

plot_walk(mlp, 'MLP calibration: gauge directions stay flat')
plot_walk(cnn, 'CNN phenomenon: extra directions stay flat')


Interpretation: the MLP is a calibration case because the known ReLU rescaling gauge explains the flat directions. The CNN is the phenomenon case: extra directions are not only small singular values; moving along them keeps loss nearly fixed for these local steps.